# Mini-Exchange Student Notebook

This is a **connectivity kit**, not a trading strategy. It shows you how to talk to the exchange: auth, prices, the order book, placing/cancelling orders, and checking your own account.

Finding a real edge (how BTC-MINI and ETH-MINI should trade relative to each other) is on you — that's the point of the exercise. Everything below is deliberately just plumbing.

**Setup:** `pip install requests`. Only need `websocket-client` if you want the streaming example at the bottom (`pip install websocket-client`).

In [3]:
import requests

# Point this at the real server for the event — ask the presenter for the
# actual host/port. Defaults here match a local dev instance.
BASE_URL = "http://178.105.55.5:8000"

# Pick your own account_id/password — this is self-serve, no admin
# approval needed, and you're active immediately with $1,000 to trade.
ACCOUNT_ID = "your_handle_here"
PASSWORD = "pick-a-password"

## 1. Register / log in

`POST /register` the first time; `POST /login` on any later run (same account_id + password gets you the same API key back). Every other endpoint needs that key in the `X-API-Key` header.

In [4]:
def register(account_id, password):
    r = requests.post(f"{BASE_URL}/register", json={"account_id": account_id, "password": password})
    r.raise_for_status()
    return r.json()["api_key"]


def login(account_id, password):
    r = requests.post(f"{BASE_URL}/login", json={"account_id": account_id, "password": password})
    r.raise_for_status()
    return r.json()["api_key"]


# First time: register(). After that, login() works too — register() on an
# already-registered account_id just returns 409, so only call it once.
API_KEY = login("ryan", "dev")
HEADERS = {"X-API-Key": API_KEY}
print("api key:", API_KEY)

api key: AkRez0t-GkNMIePfCnZyjIVKVEzp5YNm


## 2. Products and index price

`GET /products` lists both instruments, their contract size, and the current index (fair-value) price — this is *not* the same as the traded price on our book, which can and should drift from it.

In [5]:
r = requests.get(f"{BASE_URL}/products")
r.raise_for_status()
for p in r.json():
    print(f"{p['symbol']}: index=${p['index_price']:.2f}  contract_size={p['contract_size']}  max_position={p['max_position']}")

BTC-MINI: index=$78.06  contract_size=0.001  max_position=15
ETH-MINI: index=$73.79  contract_size=0.03  max_position=15


## 3. Order book snapshot

`GET /book/{product}` returns the current resting bids/asks, aggregated by price level.

In [7]:
def get_book(product):
    r = requests.get(f"{BASE_URL}/book/{product}", headers=HEADERS)
    r.raise_for_status()
    return r.json()


book = get_book("BTC-MINI")
print("bids:", book["bids"][:5])
print("asks:", book["asks"][:5])

bids: [{'price': 77.3, 'qty': 2}, {'price': 77.1, 'qty': 4}]
asks: [{'price': 78.1, 'qty': 4}]


## 4. Submit an order

`side` is `"buy"` or `"sell"`, `type` is `"limit"` or `"market"` (market = IOC, fills what it can against the book right now and drops the rest). `qty` is an integer number of contracts. Limit order prices must land on the product's tick grid — an off-grid price gets rejected.

In [10]:
def submit_order(product, side, order_type, qty, price=None):
    body = {"product": product, "side": side, "type": order_type, "qty": qty}
    if price is not None:
        body["price"] = price
    r = requests.post(f"{BASE_URL}/orders", headers=HEADERS, json=body)
    if not r.ok:
        print("rejected:", r.json().get("detail"))
        return None
    return r.json()


# Example: a resting limit buy well below the current index, so it doesn't
# immediately cross and fill — check the index price from step 2 and pick
# something sensible for your product's tick size.
order = submit_order("BTC-MINI", "buy", "limit", qty=1, price=77.60)
print(order)

{'id': 27666, 'product': 'BTC-MINI', 'side': 'buy', 'type': 'limit', 'qty': 1, 'price': 77.6, 'remaining_qty': 1, 'status': 'open'}


## 5. Check and cancel your own orders

`GET /orders` lists everything you've ever submitted (status included); `DELETE /orders/{id}` cancels one that's still open or partially filled.

In [11]:
def list_orders():
    r = requests.get(f"{BASE_URL}/orders", headers=HEADERS)
    r.raise_for_status()
    return r.json()


def cancel_order(order_id):
    r = requests.delete(f"{BASE_URL}/orders/{order_id}", headers=HEADERS)
    r.raise_for_status()
    return r.json()


open_orders = [o for o in list_orders() if o["status"] in ("open", "partially_filled")]
print("open orders:", open_orders)

if order is not None:
    print(cancel_order(order["id"]))

open orders: [{'id': 27662, 'product': 'BTC-MINI', 'side': 'buy', 'type': 'limit', 'qty': 2, 'price': 77.3, 'remaining_qty': 2, 'status': 'open'}, {'id': 27664, 'product': 'BTC-MINI', 'side': 'buy', 'type': 'limit', 'qty': 1, 'price': 77.4, 'remaining_qty': 1, 'status': 'open'}, {'id': 27666, 'product': 'BTC-MINI', 'side': 'buy', 'type': 'limit', 'qty': 1, 'price': 77.6, 'remaining_qty': 1, 'status': 'open'}]
{'id': 27666, 'product': 'BTC-MINI', 'side': 'buy', 'type': 'limit', 'qty': 1, 'price': 77.6, 'remaining_qty': 1, 'status': 'cancelled'}


## 6. Fills and account state

`GET /fills` shows your executed trades — each one tells you whether you were the `maker` or `taker` (fees differ: makers get a small rebate, takers pay), and who the counterparty was. `GET /account` gives your cash balance, positions, and realized/unrealized PnL.

In [12]:
def list_fills():
    r = requests.get(f"{BASE_URL}/fills", headers=HEADERS)
    r.raise_for_status()
    return r.json()


def get_account():
    r = requests.get(f"{BASE_URL}/account", headers=HEADERS)
    r.raise_for_status()
    return r.json()


print("recent fills:", list_fills()[:5])

acct = get_account()
print(f"balance=${acct['balance']:.2f}  equity=${acct['equity']:.2f}  positions={acct['positions']}")

recent fills: []
balance=$1000.00  equity=$1000.00  positions={}


## 7. Leaderboard

Ranked by `cash + unrealized_pnl`. No auth needed — this one's public.

In [13]:
r = requests.get(f"{BASE_URL}/leaderboard")
r.raise_for_status()
for i, row in enumerate(r.json()[:10], start=1):
    print(f"{i}. {row['account_id']}: equity=${row['equity']:.2f}")

1. ryan: equity=$1000.00
2. Caden: equity=$1000.00
3. Russell Stevens: equity=$999.96


## 8. (Optional) Streaming the book over WebSocket

`GET /book/{product}` above is a one-shot snapshot. If you want a live feed instead of polling in a loop, there's a WebSocket endpoint pushing the same snapshot every 0.5s. Needs `pip install websocket-client`.

In [ ]:
import json
import websocket  # pip install websocket-client

ws_url = BASE_URL.replace("http://", "ws://").replace("https://", "wss://")


def stream_book(product, n_messages=5):
    ws = websocket.create_connection(f"{ws_url}/book/{product}/stream?api_key={API_KEY}")
    try:
        for _ in range(n_messages):
            msg = json.loads(ws.recv())
            best_bid = msg["bids"][0]["price"] if msg["bids"] else None
            best_ask = msg["asks"][0]["price"] if msg["asks"] else None
            print(f"best bid={best_bid}  best ask={best_ask}")
    finally:
        ws.close()


stream_book("BTC-MINI", n_messages=5)

best bid=77.5  best ask=78.1
best bid=77.5  best ask=78.1
best bid=77.5  best ask=78.1
best bid=77.5  best ask=78.1
best bid=77.5  best ask=78.1


: 

## Where to go from here

That's the full connectivity surface. From here it's on you: watch how BTC-MINI and ETH-MINI move relative to each other, decide what a fair relationship between them looks like, and build whatever logic you want on top of `submit_order`/`cancel_order`.

A couple of things worth keeping in mind while you build:
- You're rate-limited per API key — a tight loop with no backoff will start getting `429` responses.
- `MAX_POSITION` caps how large a position you can hold per product — an order that would breach it gets rejected outright, not partially filled.
- Whatever edge you find, everyone else in the room can see the same two prices you can.